In [ ]:
import re
import os
import json
import datetime
import pickle
import traceback
import pandas as pd
import string

from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

In [ ]:
def generate_n_shot_examples_extraction(examples, n):
    """Takes the top n examples, flattens their messages into one list, and filters out messages with the role 'system'."""
    n_shot_examples = []
    for example in examples[:n]:
        for message in example['messages']:
            if message['role'] != 'system':  # Only add messages that don't have the 'system' role
                n_shot_examples.append(message)
    return n_shot_examples

def read_jsonl_file(file_path):
    
    """Reads a JSONL file and returns a list of examples."""
    examples = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            examples.append(json.loads(line))
    return examples

yesno_examples_file = "07_QA_YesNo_11B1-3-4_62.jsonl"     
yesno_examples = read_jsonl_file(yesno_examples_file)

factoid_examples_file = "04_QA_Factoid_11B1-3-4_76.jsonl"     
factoid_examples = read_jsonl_file(factoid_examples_file)

ideal_examples_file = "05_QA_Ideal_11B1-3-4_255.jsonl"     
ideal_examples = read_jsonl_file(ideal_examples_file)

list_examples_file = "06_QA_List_11B1-3-4_54.jsonl"     
list_examples = read_jsonl_file(list_examples_file)

def remove_punctuation_and_lowercase(text):
    # Lowercase the string
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = text[:3]
    text = text.strip()
    return text

    
def get_batch_plan(n):
    """
    根据 n (snippets数量) 返回分批方案，例如：
    4 -> [2,2], 5 -> [2,3], 7 -> [3,2,2]
    """
    if n <= 3:
        return [n]

    if n == 4:
        return [2, 2]
    if n == 5:
        return [2, 3]
    if n == 6:
        return [2, 2, 2]
    if n == 7:
        return [3, 2, 2]
    if n == 8:
        return [2, 2, 2, 2]
    if n == 9:
        return [3, 2, 2, 2]
    if n == 10:
        return [2, 2, 2, 2, 2]

    # 默认策略：尽量以 2 为主，必要时加一个 3
    plan = []
    while n > 0:
        if n % 2 == 1 and n > 3:  # 保证能拆分干净
            plan.append(3)
            n -= 3
        else:
            plan.append(2)
            n -= 2
    return plan


from openai import OpenAI
def get_completion(messages, model):
    print("Prompt Messages:")
    print(messages)
    
    # 统一使用本地 LM Studio 客户端
    client = OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="lm-studio")
    
    # 调用 API 生成对话 (全量参数按通用格式合并)
    completion = client.chat.completions.create(
        model='deepseek-r1:32b',
        # model='DeepSeek-R1-Distill-Llama-8B-exact_answer-and-ideal_answer',
        messages=messages,
        temperature=0.0
    )
    print("\n Completion:")
    print(completion)
    print("\n")
    if hasattr(completion, 'choices'):
        completion_text =  completion.choices[0].message.content
    else:
        completion_text = completion.content[0].text
    prefix = "ASSISTANT: " # bug from fireworks fine-tuning

    if completion_text.startswith(prefix):
        completion_text = completion_text[len(prefix):]
    
    print(completion_text)
    return completion_text

def generate_exact_answer(question, snippets, n_shots):
    exact_answer = []
    system_message = {"role": "system", "content": "You are BioASQ-GPT, an AI expert in question answering in the biomedical domain."}
    messages = [system_message]
    

    if question["type"] == "yesno":
        if question["lost_in_the_middle"]:
            print("⚠️ 使用bc方法")
        # """
        # 使用 MapReduce 方法回答 yes/no 问题
        # """
            few_shot_examples = generate_n_shot_examples_extraction(yesno_examples, n_shots)
            messages.extend(few_shot_examples)
        # ========= Map 阶段 =========
            map_answers = []
            # batch_size=2
            # for i in range(0, len(snippets), batch_size):
            #     batch_snippets = snippets[i:i+batch_size]
            
            # top_k=relevant_snippets
            # split=2
            # batch_size=top_k//split
            # for i in range(0, len(snippets), batch_size):
            #     batch_snippets = snippets[i:i+batch_size]
            batch_plan = get_batch_plan(len(snippets))  # 动态分批

            start = 0
            for b in batch_plan:
                batch_snippets = snippets[start:start+b]
                start += b
                map_message = {
                        "role": "user",
                        "content": f"""Based on the following biomedical evidence snippets:
                                    {batch_snippets}

                                    Question: {question['body']}

                                    Please follow these rules strictly:
                                    1. First output the lowercase word 'yes' or 'no'.
                                    2. Immediately follow it with one short sentence explaining the reason for your choice.
                                    3. If you are uncertain, you must still choose the most likely answer.

                                    Your answer:"""
                    }
                
                messages.append(map_message)
                ans = get_completion(messages, model_yesno)
                # ans = remove_punctuation_and_lowercase(clean_ideal_answer(ans))
                ans=clean_ideal_answer(ans)
                print("\nans结果：",ans)
                map_answers.append(ans)

            print("\nMap 阶段结果:", map_answers)

            reduce_message = [
                {
                    "role": "user",
                    "content": f"""以下是对同一问题的多个分段回答：{map_answers}

                    问题：{question['body']}

                    请严格按以下规则给出最终回答：
                    1. 仅允许输出小写单词 'yes' 或 'no'
                    2. 禁止添加任何解释、标点或格式修饰
                    3. 如果所有答案一致，则直接采纳该答案
                    4. 如果答案存在分歧，请优先选择与问题语义最契合、最合理的答案，而不是简单多数投票
                    5. 如果只有一个回答，也需基于其合理性判断是否作为最终答案

                    你的最终回答："""
                }
            ]

            final_answer = get_completion(reduce_message, model_yesno)
            final_answer = remove_punctuation_and_lowercase(clean_ideal_answer(final_answer))
            print("\nReduce 阶段最终答案:", final_answer)
        else:
            few_shot_examples = generate_n_shot_examples_extraction(yesno_examples, n_shots)
            messages.extend(few_shot_examples)
            # user_message = {"role": "user", "content": f"""{snippets}\n\n
            #         '{question['body']}'. 
            #          You *must answer* only with lowercase 'yes' or 'no' even if you are not sure about the answer."""}
            user_message = {
                            "role": "user",
                            "content": f"""基于以下医学资料片段：
                            {snippets}

                            问题：{question['body']}

                            请严格按以下规则回答：
                            1. 仅允许输出小写单词 'yes' 或 'no'
                            2. 禁止添加任何解释、标点或格式修饰
                            3. 不确定时仍必须选择最可能的答案

                            你的回答："""  # 明确输出格式
                            }
            messages.append(user_message)
            print(messages)
            answer = get_completion(messages, model_yesno)
            print("\ngpt response yesno:")
            clean_answer=clean_ideal_answer(answer)
            print(clean_answer)
            final_answer = remove_punctuation_and_lowercase(clean_answer) 
        return final_answer
    elif question["type"] == "factoid":
        if question["lost_in_the_middle"]:
            print("⚠️ 使用bc方法")
            few_shot_examples = generate_n_shot_examples_extraction(factoid_examples, n_shots)

            # ========= Map 阶段 =========
            map_results = []
            # batch_size = 2  # 每次处理几个 snippet，可根据需求调整
            # for i in range(0, len(snippets), batch_size):
            #     batch_snippets = snippets[i:i+batch_size]
            
            # top_k=relevant_snippets
            # split=2
            # batch_size=top_k//split
            # for i in range(0, len(snippets), batch_size):
            #     batch_snippets = snippets[i:i+batch_size]
            batch_plan = get_batch_plan(len(snippets))  # 动态分批

            start = 0
            for b in batch_plan:
                batch_snippets = snippets[start:start+b]
                start += b
                
                map_message = []
                map_message.extend(few_shot_examples)
                                
                map_message.append({
                    "role": "user",
                    "content": f"""
                    基于以下医学资料片段：
                    {batch_snippets}`   

                    问题：'{question['body']}'

                    请严格按照以下要求回答：
                    1. 返回一个仅包含 "entities" 键的 JSON 对象。
                    2. "entities" 的值必须是一个数组，包含 0-5 个候选答案。
                    3. 候选答案要求：
                    - 必须直接回答问题（如疾病名、基因名、数值、药物名、解剖结构等）。
                    - 禁止返回解释性短语、定义或完整句子。
                    - 如果存在常见的等价表述（如全称/缩写），可以同时返回。
                    - 按置信度从高到低排序。
                    4. JSON 必须严格有效：
                    - 使用双引号
                    - 无额外文本或注释
                    - 可被标准 JSON 解析器正确解析
                    5. 如果无法确定答案，返回空数组：{{"entities": []}}

                    示例1：
                    问题：What is the use of lebrikizumab?
                    响应：{{"entities": ["atopic dermatitis treatment"]}}

                    示例2：
                    问题：Coffee Bean Sign is characteristic to which disease?
                    响应：{{"entities": ["sigmoid volvulus"]}}
                    """

                })

                raw_answer = get_completion(map_message, model_factoid)

                # 提取 JSON
                def extract_json_from_response(text):
                    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
                    text = text.replace('```json', '').replace('```', '')
                    json_match = re.search(r'\{.*\}', text, re.DOTALL)
                    return json_match.group(0) if json_match else text.strip()

                cleaned = extract_json_from_response(raw_answer)
                try:
                    parsed = json.loads(cleaned)
                    entities = parsed.get("entities", [])
                    if isinstance(entities, list):
                        # 转成字符串并去除 None
                        map_results.extend([str(e).strip() for e in entities if isinstance(e, (str, int, float))])
                except:
                    continue

            print("Map 阶段结果:", map_results)

            # ========= Reduce 阶段 =========
            reduce_messages = []
            reduce_messages.extend(few_shot_examples)
            reduce_messages.append({
                "role": "user",
                "content": f"""以下是候选实体列表（可能重复或包含噪音）：
        {map_results}

        问题：'{question['body']}'

        请严格按以下要求给出最终答案：
        
        1. 仅对候选实体去重和规范化，不要合并或抽象
        2. 按出现频率或置信度从高到低排序，最多保留5个
        3. 返回一个JSON对象，格式：
        {{"entities": ["Answer 1", "Answer 2"]}}
        4. 每个实体必须与候选实体中出现过的内容保持一致，不能自己生成新答案
        5. 禁止添加任何解释文字，只输出JSON"""
            })

            reduce_answer = get_completion(reduce_messages, model_factoid)

            cleaned_reduce = extract_json_from_response(reduce_answer)
            try:
                parsed_final = json.loads(cleaned_reduce)
                valid_entities = parsed_final.get("entities", [])
                exact_answer = [valid_entities] if valid_entities else []
            except:
                exact_answer = []

            print("Reduce 阶段最终答案:", exact_answer)
        else:
            print("⚠️ 使用原方法")
                # 获取n-shot 示例
            few_shot_examples = generate_n_shot_examples_extraction(factoid_examples, n_shots)
            messages.extend(few_shot_examples)

            user_message = {
                "role": "user", 
                "content": f"""
                            {snippets}

                            问题：'{question['body']}'

                            请严格按照以下要求回答：
                            1. 返回一个仅包含 "entities" 键的 JSON 对象。
                            2. "entities" 的值必须是一个数组，包含 0-5 个候选答案。
                            3. 候选答案要求：
                            - 必须直接回答问题（如疾病名、基因名、数值、药物名、解剖结构等）。
                            - 禁止返回解释性短语、定义或完整句子。
                            - 如果存在常见的等价表述（如全称/缩写），可以同时返回。
                            - 按置信度从高到低排序。
                            4. JSON 必须严格有效：
                            - 使用双引号
                            - 无额外文本或注释
                            - 可被标准 JSON 解析器正确解析
                            5. 如果无法确定答案，返回空数组：{{"entities": []}}

                            示例1：
                            问题：What is the use of lebrikizumab?
                            响应：{{"entities": ["atopic dermatitis treatment"]}}

                            示例2：
                            问题：Coffee Bean Sign is characteristic to which disease?
                            响应：{{"entities": ["sigmoid volvulus"]}}
                            """

                            }
            
            messages.append(user_message)
            answer = get_completion(messages, model_factoid)
            print("初始响应:", answer)

            # 新增清洗步骤
            def extract_json_from_response(text):
                # 去除think标签
                text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
                # 去除可能的代码块标记
                text = text.replace('```json', '').replace('```', '')
                # 提取第一个出现的JSON对象
                json_match = re.search(r'\{.*\}', text, re.DOTALL)
                return json_match.group(0) if json_match else text.strip()

            cleaned_answer = extract_json_from_response(answer)
            print("清洗后响应:", cleaned_answer)

            try:
                parsed = json.loads(cleaned_answer)
                
                # 增强校验逻辑
                if not isinstance(parsed, dict):
                    raise ValueError("响应不是字典格式")
                    
                raw_entities = parsed.get("entities", [])
                
                # 类型转换和过滤
                valid_entities = []
                for item in raw_entities[:5]:  # 先截断再处理
                    if isinstance(item, (str, int, float)):
                        valid_entities.append(str(item).strip())
                    elif item is None:  # 过滤空值
                        continue
                        
                print("有效实体列表:", valid_entities)
                
            except (json.JSONDecodeError, ValueError, KeyError) as e:
                print(f"解析失败: {str(e)}")
                valid_entities = []

            exact_answer = [valid_entities] if valid_entities else []
            # exact_answer = [[entity] for entity in valid_entities]
            print("最终exact_answer:", exact_answer)
    elif question["type"] == "list":
        if question["lost_in_the_middle"]:
            print("⚠️ 使用bc方法")
            try:
                few_shot_examples = generate_n_shot_examples_extraction(list_examples, n_shots)
                # ========= Map 阶段 =========
                map_entities = []
                # batch_size = 2  # 每次处理几个 snippet，可调整
                # for i in range(0, len(snippets), batch_size):
                #     batch_snippets = snippets[i:i+batch_size]
                
                # top_k=relevant_snippets
                # split=2
                # batch_size=top_k//split
                # for i in range(0, len(snippets), batch_size):
                #     batch_snippets = snippets[i:i+batch_size]
                batch_plan = get_batch_plan(len(snippets))  # 动态分批

                start = 0
                for b in batch_plan:
                    batch_snippets = snippets[start:start+b]
                    start += b
                
                    map_message = []
                    map_message.extend(few_shot_examples)
                                # 2. 实体必须直接来自文献片段，不得凭空生成。
                    map_message.append({
                        "role": "user",
                        "content": f"""
                基于以下医学资料片段：
                {batch_snippets}

                问题：{question['body']}

                请严格按以下要求回答：
                1. 返回严格符合此格式的JSON：{{"entities": ["实体1", "实体2"]}}
                2. 实体必须与问题语义严格对应：
                - 如果问题询问“疾病/综合征”，只返回疾病/综合征名称。
                - 如果问题询问“基因/突变”，只返回基因或突变名称。
                - 如果问题询问“方法/手术/治疗”，只返回方法或手术名称。
                - 如果问题询问“症状/体征”，只返回症状或体征。
                - 如果问题询问“药物/成分/分子”，只返回药物名或活性成分。
                - 如果问题询问“来源”，只返回生物学或组织来源。
                - 如果问题询问“受体/靶点”，只返回分子受体或通路。
                3. 返回的实体必须与问题要求的层次一致（不要列出更低层级的步骤、功能或作用机制）。
                4. 按文献中出现频率排序（高频在前）。
                5. 每个实体必须是简洁的名词短语（2-5个单词）。
                6. 最多返回100个不同实体。
                7. 输出中不要包含任何解释性文字。

                [示例1]
                问题：What are the common symptoms of COVID-19?
                响应：{{"entities": ["fever", "cough", "fatigue", "loss of smell"]}}

                [示例2]
                问题：List the methods for pectus excavatum repair?
                响应：{{"entities": ["Nuss procedure", "Ravitch procedure", "sternotomy", "vacuum bell"]}}
                """

                    })
                    
                    raw_answer = get_completion(map_message, model_list)

                    # 提取 JSON 实体
                    def extract_json_entities(text):
                        text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
                        text = text.replace('```json', '').replace('```', '')
                        json_match = re.search(r'\{.*\}', text, re.DOTALL)
                        if json_match:
                            try:
                                parsed = json.loads(json_match.group(0))
                                return parsed.get("entities", [])
                            except:
                                return []
                        # fallback 从任意文本中提取实体
                        return re.findall(r'"([\w\s-]+)"', text)

                    batch_entities = extract_json_entities(raw_answer)
                    map_entities.extend(batch_entities)

                print("Map 阶段结果:", map_entities)

                # ========= Reduce 阶段 =========
                reduce_message = []
                reduce_message.extend(few_shot_examples)
                reduce_message.append({
                    "role": "user",
                    "content": f"""
        以下是候选实体列表（可能重复或包含噪音）：
        {map_entities}

        问题：{question['body']}

        请严格按以下要求给出最终答案：
        1. 对候选实体仅做去重和规范化，不要概括、合并或抽象
        2. 按出现频率从高到低排序，保留前 100 个
        3. 返回严格符合 JSON 格式：{{"entities": ["实体1", "实体2"]}}
        4. 每个实体必须与候选实体中出现过的内容保持一致，不能自己生成新实体
        5. 禁止添加任何解释文字，只输出 JSON
        """
                })

                reduce_answer = get_completion(reduce_message, model_list)
                final_entities = extract_json_entities(reduce_answer)

                # 去重且保留顺序，最多100个
                seen = set()
                filtered_entities = []
                for e in final_entities:
                    if e not in seen and 1 <= len(e.split()) <= 5:
                        seen.add(e)
                        filtered_entities.append(e)
                    if len(filtered_entities) >= 100:
                        break

                exact_answer = [[e] for e in filtered_entities]
                print("Reduce 阶段最终答案:", exact_answer)

            except Exception as e:
                print(f"列表问题处理异常: {str(e)}")
                exact_answer = []
        else:
            print("⚠️ 使用原方法")
            try:
                # ================= 1. 准备提示词 =================
                few_shot_examples = generate_n_shot_examples_extraction(list_examples, n_shots)
                # messages = [
                #     {"role": "system", "content": "您是一个生物医学信息检索专家，严格按照要求格式返回答案"},
                messages = [
                    *few_shot_examples,
                    {
                        "role": "user", 
                        "content": f"""
                        基于以下医学资料片段：
                        {snippets}\n\n
                        [问题]：{question['body']}
                        
                        [要求]：
                        1. 返回严格符合此格式的JSON：{{"entities": ["实体1", "实体2"]}}
                        2. 实体必须直接来自文献片段，不得凭空生成。
                        3. 实体必须与问题语义严格对应：
                        - 如果问题询问“疾病/综合征”，只返回疾病/综合征名称。
                        - 如果问题询问“基因/突变”，只返回基因或突变名称。
                        - 如果问题询问“方法/手术/治疗”，只返回方法或手术名称。
                        - 如果问题询问“症状/体征”，只返回症状或体征。
                        - 如果问题询问“药物/成分/分子”，只返回药物名或活性成分。
                        - 如果问题询问“来源”，只返回生物学或组织来源。
                        - 如果问题询问“受体/靶点”，只返回分子受体或通路。
                        4. 返回的实体必须与问题要求的层次一致（不要列出更低层级的步骤、功能或作用机制）。
                        5. 按文献中出现频率排序（高频在前）。
                        6. 每个实体必须是简洁的名词短语（2-5个单词）。
                        7. 最多返回100个不同实体。
                        8. 输出中不要包含任何解释性文字。

                        [示例1]
                        问题：What are the common symptoms of COVID-19?
                        响应：{{"entities": ["fever", "cough", "fatigue", "loss of smell"]}}

                        [示例2]
                        问题：List the methods for pectus excavatum repair?
                        响应：{{"entities": ["Nuss procedure", "Ravitch procedure", "sternotomy", "vacuum bell"]}}
                        """
                    }
                ]
                
                # ================= 2. 获取模型响应 =================
                raw_answer = get_completion(messages, model_list)
                print("\n模型原始响应:", raw_answer)

                # ================= 3. 强化版响应处理 =================
                def process_list_response(response):
                    # 匹配严格JSON结构
                    json_match = re.search(r'\{\s*"entities"\s*:\s*$$.*?$$\s*\}', response, re.DOTALL)
                    
                    if json_match:
                        # 提取并修复JSON
                        json_str = json_match.group(0)
                        # 替换中文符号
                        json_str = (json_str
                                .replace("“", '"').replace("”", '"')
                                .replace("‘", "'").replace("’", "'")
                                .replace("：", ":").replace("，", ","))
                        try:
                            parsed = json.loads(json_str)
                            if isinstance(parsed, dict):
                                return parsed.get("entities", [])
                        except json.JSONDecodeError:
                            pass
                    
                    # 应急处理：从任意文本中提取实体
                    entities = re.findall(r'"([\w\s-]+)"', response)  # 匹配带引号的实体
                    return [e for e in entities if e != "entities"]   # 过滤键名

                # 执行处理流程
                raw_entities = process_list_response(raw_answer)
                
                # ================= 4. 数据清洗 =================
                # 去重且保留顺序
                seen = set()
                filtered_entities = []
                for entity in raw_entities:
                    if entity not in seen and 1 <= len(entity.split()) <= 5:
                        seen.add(entity)
                        filtered_entities.append(entity)
                    if len(filtered_entities) >= 100:
                        break

                # ================= 5. 最终格式化 =================
                exact_answer = [[e] for e in filtered_entities]
                print("处理后的列表答案:", exact_answer)

            except Exception as e:
                print(f"列表问题处理异常: {str(e)}")
                exact_answer = []
    return exact_answer

def clean_ideal_answer(text):
    # 使用正则表达式删除<think>标签及其内容
    cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    # 去除可能残留的换行符并重新整理格式
    cleaned = re.sub(r'\n{2,}', '\n\n', cleaned.strip())
    
    return cleaned

def generate_ideal_answer(question, snippets, n_shots):
    system_message = {"role": "system", "content": "​You are a biomedical expert. Please provide concise and professional answers directly, without including your thought process."}
    messages = [system_message]
    few_shot_examples = generate_n_shot_examples_extraction(ideal_examples, n_shots)
    messages.extend(few_shot_examples)
    user_message = {"role": "user", "content": f""" {snippets}\n\n
             '{question['body']}'.
             You are a biomedical expert, write a concise and clear answer to the above question.
             It is very important that the answer is correct.
             The maximum allowed length of the answer is 200 words, but try to keep it short and concise."""}
    messages.append(user_message)
    print(messages)
    answer = get_completion(messages, model_ideal)
    print("\ngpt response ideal:")
    
    # print(answer)
    clean_answer = clean_ideal_answer(answer)
    print(clean_answer)
    return clean_answer    



# Load the input file in JSON format
with open('/workspace/code/bioasq2024/02_12B/Batch1/PhaseA+/phaseB_12b_01-yujian.json', encoding='utf-8') as input_file:
    data = json.loads(input_file.read())

model_name = "deepseek-r1:32b"
model_yesno = "deepseek-r1:32b"
model_ideal = "deepseek-r1:32b"


model_list = "deepseek-r1:32b"
model_factoid = "deepseek-r1:32b"




n_shots = 10


# Get the current timestamp in a sortable format
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

if '/' in model_name or ':' in model_name:
    pickl_name = model_name.replace('/', '-').replace(':', '-')
else:
    pickl_name = model_name
pickl_file = f'deepseek-r1-32b-10-shot.pkl'



def save_state(data, file_path=pickl_file):
    """Save the current state to a pickle file."""
    with open(file_path, 'wb') as f:
        pickle.dump(data, f)

def load_state(file_path=pickl_file):
    
    
    """Load the state from a pickle file if it exists, otherwise return None."""
    try:
        if os.path.exists(file_path):
            with open(file_path, 'rb') as f:
                return pickle.load(f)
    except EOFError:  # Handles empty pickle file scenario
        return None
    return None

# Define columns
columns = ['id', 'body', 'type', 'documents', 'snippets', "ideal_answer", "exact_answer"]

# Initialize empty DataFrame
questions_df = pd.DataFrame(columns=columns)

saved_df = load_state(pickl_file)


if saved_df is not None and not saved_df.empty:
    processed_ids = set(saved_df['id'])  
    questions_df = saved_df
else:
    processed_ids = set()

questions_to_process = [q for q in data["questions"] if q["id"] not in processed_ids]
#questions_to_process = questions_to_process[:2]


def process_question(question):
    ideal_answer = []
    question_type = question["type"]
    print(f"{question['body']}\n")

    # Get the relevant articles and snippets
    relevant_snippets = question["snippets"]

    # Generate the exact answer and ideal answer
    try:
        exact_answer = generate_exact_answer(question, relevant_snippets, n_shots)
        # ideal_answer = generate_ideal_answer(question, relevant_snippets, n_shots)
    except Exception as e:
        print(f"Error processing question {question['id']}: {e}")
        traceback.print_exc()
        exact_answer = []
        ideal_answer = []


    # Create a dictionary to store the results for this question
    question_results = {
        "id": question["id"],
        "type": question_type,
        "body": question["body"],
        "documents": question["documents"],
        "snippets": question["snippets"],
        "ideal_answer": ideal_answer,
        "exact_answer": exact_answer,
    }
    return question_results

# Use ThreadPoolExecutor to process questions in parallel
with ThreadPoolExecutor(max_workers=4) as executor:
    # Dictionary to keep track of question futures
    future_to_question = {executor.submit(process_question, q): q for q in questions_to_process}
    
    for future in as_completed(future_to_question):
        question = future_to_question[future]
        try:
            result = future.result()
            if result:
                # Append result to the DataFrame
                result_df = pd.DataFrame([result])
                questions_df = pd.concat([questions_df, result_df], ignore_index=True)
                save_state(questions_df, pickl_file)
        except Exception as e:
            print(f"Error processing question {question['id']}: {e}")
            traceback.print_exc()


# Prefix the output file name with the timestamp
if '/' in model_name:
    model_name_pretty = model_name.split("/")[-1]
else:
    model_name_pretty = model_name
output_file_name = f"./Results/{timestamp}_{model_name_pretty}_11B2-{n_shots}-QA.csv"

# Ensure the directory exists before saving
os.makedirs(os.path.dirname(output_file_name), exist_ok=True)

questions_df.to_csv(output_file_name, index=False)

# After processing all questions and saving the final output:
try:\
    # Check if the pickle file exists before attempting to delete it
    if os.path.exists(pickl_file):
        
        
        os.remove(pickl_file)
        print("Intermediate state pickle file deleted successfully.")
except Exception as e:
    print(f"Error deleting pickle file: {e}")
    traceback.print_exc()

## Run File Generation

In [ ]:
import pandas as pd
import json

def csv_to_json(csv_filepath, json_filepath):
    # Step 1: Read the CSV file into a pandas DataFrame
    df = pd.read_csv(csv_filepath)
    
    # Transform the DataFrame into a list of dictionaries, one per question
    questions_list = df.to_dict(orient='records')
    
    # Initialize the structure of the JSON file
    json_structure = {"questions": []}
    
    # Step 2: Transform the DataFrame into the desired JSON structure
    for item in questions_list:
        question_dict = {
            "documents": eval(item["documents"])[:10],
            "snippets": eval(item["snippets"])[:10],
            "body": item["body"],
            "type": item["type"],
            "id": item["id"],
            "ideal_answer": item["ideal_answer"],
        }
        if item["type"] == "yesno":
            yesno_answer = item["exact_answer"]
            if yesno_answer not in ['yes', 'no']:
                print(yesno_answer)
                yesno_answer = 'no'
            question_dict["exact_answer"] = yesno_answer
        # if item["type"] == "factoid":
        #     question_dict["exact_answer"] = eval(item["exact_answer"])[:5]
        elif item["type"] == "factoid":
            question_dict["exact_answer"] = eval(item["exact_answer"])[:5]
            # question_dict["exact_answer"] = []
        if item["type"] == "list":
            question_dict["exact_answer"] = eval(item["exact_answer"])[:100]
            
            

        json_structure["questions"].append(question_dict)
    
    # Step 3: Write the JSON structure to a file
    with open(json_filepath, 'w', encoding='utf-8') as json_file:
        json.dump(json_structure, json_file, ensure_ascii=False, indent=4)


# Example usage  

csv_filepath = '/workspace/code/bioasq2024/02_12B/Batch1/PhaseA+/Results/2026-04-30_15-14-34_deepseek-r1:32b_11B2-10-QA.csv'  # Update this path to your actual CSV file path
json_filepath = '/workspace/code/bioasq2024/02_12B/Batch1/PhaseA+/Results/12b-Batch4-Phase-B-deepseek-32b-yujian.json'  # Update this path to where you want to save the JSON file
csv_to_json(csv_filepath, json_filepath)
